In [7]:
# Install Kaggle API package
!pip install -q kaggle

# Download the dataset directly from Kaggle
!kaggle datasets download -d pranavraikokte/covid19-image-dataset
!unzip -q covid19-image-dataset.zip -d dataset

print("Dataset downloaded and extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset
License(s): CC-BY-SA-4.0
100% 158M/158M [00:02<00:00, 82.7MB/s]

Dataset downloaded and extracted successfully!


In [8]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define dataset paths
train_dir = 'dataset/Covid19-dataset/train'
test_dir = 'dataset/Covid19-dataset/test'
IMG_SIZE = 128

def load_images_and_labels(base_dir):
    images = []
    labels = []
    # Automatically read subfolder names to prevent path/casing errors
    categories = sorted(os.listdir(base_dir))
    categories = [cat for cat in categories if os.path.isdir(os.path.join(base_dir, cat))]
    print(f"Found Categories in {base_dir}: {categories}")

    for category in categories:
        path = os.path.join(base_dir, category)
        class_num = categories.index(category)
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                    images.append(img)
                    labels.append(class_num)
            except Exception as e:
                pass

    return np.array(images), np.array(labels), categories

X_train, y_train, class_names = load_images_and_labels(train_dir)
X_test, y_test, _ = load_images_and_labels(test_dir)

# Normalize pixel values to range [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"\nTraining Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")
print(f"Class Mapping: {class_names}")

Found Categories in dataset/Covid19-dataset/train: ['Covid', 'Normal', 'Viral Pneumonia']
Found Categories in dataset/Covid19-dataset/test: ['Covid', 'Normal', 'Viral Pneumonia']

Training Data Shape: (251, 128, 128, 3)
Testing Data Shape: (66, 128, 128, 3)
Class Mapping: ['Covid', 'Normal', 'Viral Pneumonia']


In [9]:
# Build a clean, lightweight CNN architecture perfect for web apps
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(class_names), activation='softmax') # Dynamically matches number of classes
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=32
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,027 (12.61 MB)

 Trainable params: 3,305,027 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.5300 - loss: 1.0919 - val_accuracy: 0.0000e+00 - val_loss: 1.7308
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.6700 - loss: 0.8141 - val_accuracy: 0.0000e+00 - val_loss: 1.6789
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.8300 - loss: 0.5423 - val_accuracy: 0.2157 - val_loss: 1.4814
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 996ms/step - accuracy: 0.9100 - loss: 0.2256 - val_accuracy: 0.5882 - val_loss: 1.1228
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 867ms/step - accuracy: 0.9200 - loss: 0.1811 - val_accuracy: 0.6863 - val_loss: 0.8639
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9600 - loss: 0.1186 - val_accuracy: 0.7647 - val_loss: 0.7046
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 869ms/step - accuracy: 0.9850 - loss: 0.0804 - val_accuracy: 0.7451 - val_loss: 0.8662
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 833ms/step - accuracy: 0.9600 - loss: 0.1118 - val_accuracy: 0.6863 - val_loss: 

In [14]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

# Save the trained model file to your Colab environment
model.save("covid_model.h5", include_optimizer=False)
print("Model saved successfully as 'covid_model.h5'!")

# Download the model file to your local computer
from google.colab import files
files.download('covid_model.h5')

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step - accuracy: 0.7879 - loss: 0.5745



Test Accuracy: 78.79%
Model saved successfully as 'covid_model.h5'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>